# Aula 1 — Complemento: agentes e ferramentas controladas

## Caso condutor: triagem de incidentes a partir de um identificador de ticket

No notebook principal da Aula 1, o código Python chamava as funções de consulta antes da chamada ao modelo. Isso produzia um **workflow linear e determinístico**:

```text
código → busca ticket/logs/status → monta prompt → LLM → resposta
```

Este complemento altera especificamente **quem decide consultar uma fonte de dados**. Agora, o agente recebe uma ferramenta chamada `buscar_ticket` e pode escolhê-la durante a conversa:

```text
usuário → agente → decisão de usar ferramenta → buscar_ticket → agente → resposta final
```

A diferença é de controle do fluxo, não de permissões. O agente **não ganha acesso amplo** à base de tickets: ele pode executar apenas a operação explicitamente autorizada, com um parâmetro e uma resposta bem definidos.

> **Objetivo didático:** distinguir um workflow enriquecido por contexto de um agente que chama ferramentas em um ciclo de decisão e execução.

---

### O que este notebook cobre

1. uma base pequena de tickets e funções puras de consulta;
2. um modo demonstrativo, sem API, que torna visível o ciclo de ferramenta;
3. uma implementação opcional de agente com `LangChain` e `@tool`;
4. como representar a mesma arquitetura no canvas do Langflow;
5. limites, validações e exercícios para não confundir autonomia com acesso irrestrito.


## 1. Setup e modos de execução

O notebook foi desenhado para ser **independente** do notebook principal.

- **Modo demonstração:** roda apenas com a biblioteca padrão do Python. Ele simula de forma determinística a decisão de chamar uma ferramenta. É útil para explicar o mecanismo sem chave de API, mas **não é um LLM agente real**.
- **Modo agente real:** usa `LangChain`, um modelo compatível com OpenAI e uma chave de API. Nesse caso, o modelo recebe a descrição da ferramenta e decide se deve chamá-la.

Para executar a parte opcional, instale as dependências em seu ambiente:

```bash
pip install langchain langchain-openai python-dotenv
```

E crie um arquivo `.env` com, no mínimo:

```text
OPENAI_API_KEY=sua_chave
OPENAI_MODEL=nome_do_modelo
```

O nome do modelo não é fixado no material: use um modelo habilitado em sua conta e com suporte a tool calling.


In [ ]:
load_dotenv()

USE_DEMO_MODE = os.getenv("USE_DEMO_MODE")
OPENAI_MODEL = os.getenv("OPENAI_MODEL")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("Dados em:", DATA_DIR.resolve())
print("Modo demonstração:", USE_DEMO_MODE)
print("Modelo configurado:", OPENAI_MODEL)


Dados em: C:\Users\cstefano\Desktop\Carlos\Alura\Skills and Go\Aulas\agentic_engineering_aula1_notebook_first_llm\agentic_engineering_aula1_notebook_first_llm\data
Modo demonstração: false
Modelo configurado: gpt-4.1-mini


In [7]:
from __future__ import annotations

import os
import re
from dataclasses import dataclass, asdict
from typing import Any, Callable

try:
    from dotenv import load_dotenv
    load_dotenv(dotenv_path=".env", override=False)
except ImportError:
    pass

USE_DEMO_MODE = os.getenv("USE_DEMO_MODE", "true").strip().lower() in {"1", "true", "yes"}
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "")

print(f"Modo demonstração: {USE_DEMO_MODE}")
print(f"Modelo configurado: {OPENAI_MODEL or 'não configurado'}")


Modo demonstração: False
Modelo configurado: gpt-4.1-mini


## 2. Base de dados didática e funções puras

Para manter o notebook independente, a base de tickets fica embutida no próprio arquivo. Em um sistema real, ela poderia ser substituída por uma API, banco de dados ou sistema de chamados.

O ponto importante é que a regra de negócio continua sendo funcional:

- recebe dados explícitos como argumentos;
- devolve dados estruturados;
- não depende de estado oculto;
- não deixa o modelo executar consultas genéricas.

A estrutura abaixo também limita deliberadamente o que pode ser recuperado. Não incluímos, por exemplo, dados pessoais, credenciais, histórico integral do cliente ou capacidade de alterar o ticket.


In [8]:
TICKETS = {
    "INC-2026-001": {
        "ticket_id": "INC-2026-001",
        "title": "Falhas de login após atualização",
        "service": "authentication",
        "description": "Clientes relatam erro 500 e timeout ao tentar entrar na plataforma.",
        "priority": "high",
        "status": "open",
    },
    "INC-2026-002": {
        "ticket_id": "INC-2026-002",
        "title": "Cobrança duplicada em pagamentos",
        "service": "billing",
        "description": "Alguns clientes identificaram duas cobranças para a mesma transação.",
        "priority": "medium",
        "status": "open",
    },
    "INC-2026-003": {
        "ticket_id": "INC-2026-003",
        "title": "Lentidão no relatório semanal",
        "service": "analytics",
        "description": "A geração do relatório semanal está mais lenta que o esperado, sem indisponibilidade.",
        "priority": "low",
        "status": "open",
    },
}


def normalizar_ticket_id(ticket_id: str) -> str:
    """Normaliza o identificador antes da consulta."""
    return ticket_id.strip().upper()


def ticket_nao_encontrado(ticket_id: str) -> dict[str, str]:
    """Representa uma falha de consulta como dado estruturado, não como texto solto."""
    return {
        "error": "ticket_not_found",
        "ticket_id": ticket_id,
    }


def buscar_ticket(ticket_id: str, tickets: dict[str, dict[str, str]] = TICKETS) -> dict[str, str]:
    """
    Recupera os campos autorizados de um ticket pelo identificador.

    Esta é a capacidade que será exposta ao agente. Ela não aceita SQL,
    filtros arbitrários ou comandos de escrita; aceita apenas um ticket_id.
    """
    ticket_id_normalizado = normalizar_ticket_id(ticket_id)
    ticket = tickets.get(ticket_id_normalizado)

    if ticket is None:
        return ticket_nao_encontrado(ticket_id_normalizado)

    return dict(ticket)


buscar_ticket("inc-2026-002")


{'ticket_id': 'INC-2026-002',
 'title': 'Cobrança duplicada em pagamentos',
 'service': 'billing',
 'description': 'Alguns clientes identificaram duas cobranças para a mesma transação.',
 'priority': 'medium',
 'status': 'open'}

## 3. Antes do agente: o workflow linear do notebook principal

No padrão anterior, o **código da aplicação** decide que precisa consultar o ticket. O LLM recebe o ticket já recuperado no prompt.

Esse desenho costuma ser melhor quando você sabe antecipadamente quais dados são necessários. Ele é simples, previsível, barato e mais fácil de testar.

A função abaixo representa apenas o ponto de orquestração. Ela consulta o ticket sempre, independentemente da pergunta do usuário.


In [9]:
def montar_contexto_deterministico(ticket_id: str) -> dict[str, Any]:
    ticket = buscar_ticket(ticket_id)

    return {
        "ticket_consultado_pelo_codigo": True,
        "ticket": ticket,
    }


montar_contexto_deterministico("INC-2026-001")


{'ticket_consultado_pelo_codigo': True,
 'ticket': {'ticket_id': 'INC-2026-001',
  'title': 'Falhas de login após atualização',
  'service': 'authentication',
  'description': 'Clientes relatam erro 500 e timeout ao tentar entrar na plataforma.',
  'priority': 'high',
  'status': 'open'}}

## 4. O que muda quando introduzimos uma ferramenta

Uma ferramenta é uma função com três elementos que o modelo consegue conhecer:

1. **nome**: por exemplo, `buscar_ticket`;
2. **descrição**: quando ela deve ser usada;
3. **schema de entrada**: quais argumentos aceita e em qual formato.

O modelo não executa Python diretamente. Ele devolve uma solicitação estruturada de chamada de ferramenta. A aplicação — ou o framework — valida e executa a função permitida, devolvendo o resultado ao modelo.

```text
1. Usuário: “Analise o ticket INC-2026-002.”
2. Modelo: “Preciso consultar buscar_ticket(ticket_id=INC-2026-002).”
3. Aplicação: executa buscar_ticket(...).
4. Ferramenta: devolve os dados autorizados.
5. Modelo: produz a análise final usando o resultado.
```

A autonomia é, portanto, **limitada à seleção entre capacidades autorizadas**.


In [10]:
@dataclass(frozen=True)
class ToolCall:
    name: str
    arguments: dict[str, str]


@dataclass(frozen=True)
class ToolResult:
    name: str
    output: dict[str, str]


def descrever_ferramenta_buscar_ticket() -> dict[str, Any]:
    """Metadados equivalentes aos que um framework fornece ao modelo."""
    return {
        "name": "buscar_ticket",
        "description": (
            "Recupera detalhes de um ticket de suporte a partir de seu identificador. "
            "Use quando for necessário conhecer título, serviço, descrição, prioridade ou status."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "ticket_id": {
                    "type": "string",
                    "description": "Identificador do ticket, como INC-2026-001.",
                }
            },
            "required": ["ticket_id"],
            "additionalProperties": False,
        },
    }


descrever_ferramenta_buscar_ticket()


{'name': 'buscar_ticket',
 'description': 'Recupera detalhes de um ticket de suporte a partir de seu identificador. Use quando for necessário conhecer título, serviço, descrição, prioridade ou status.',
 'input_schema': {'type': 'object',
  'properties': {'ticket_id': {'type': 'string',
    'description': 'Identificador do ticket, como INC-2026-001.'}},
  'required': ['ticket_id'],
  'additionalProperties': False}}

## 5. Modo demonstração: visualizando o ciclo do agente sem API

Esta parte **não usa um LLM**. A função `decidir_proxima_acao_demo` simula, por regras transparentes, a decisão que um modelo com tool calling tomaria.

Ela existe por duas razões didáticas:

- permite que toda a turma execute e observe o fluxo sem custo ou chave de API;
- separa a arquitetura do ciclo de ferramentas da qualidade variável de uma resposta gerada por modelo.

Em produção, essa função de decisão será substituída pelo modelo, como veremos na Seção 6.


In [11]:
def extrair_ticket_id(texto: str) -> str | None:
    """Extrai um identificador no formato INC-AAAA-NNN de uma mensagem."""
    padrao = r"\bINC-\d{4}-\d{3}\b"
    correspondencia = re.search(padrao, texto.upper())
    return correspondencia.group(0) if correspondencia else None


def decidir_proxima_acao_demo(mensagem_usuario: str, contexto: list[dict[str, Any]]) -> ToolCall | None:
    """
    Simula a decisão do agente.

    Retorna uma ToolCall quando há um ticket mencionado e ainda não existe
    resultado de consulta no contexto. Caso contrário, indica que o ciclo
    pode terminar com uma resposta final.
    """
    ticket_id = extrair_ticket_id(mensagem_usuario)
    ja_consultou_ticket = any(item.get("type") == "tool_result" for item in contexto)

    if ticket_id and not ja_consultou_ticket:
        return ToolCall(name="buscar_ticket", arguments={"ticket_id": ticket_id})

    return None


def executar_tool_call(tool_call: ToolCall) -> ToolResult:
    """Executor controlado: apenas nomes de ferramentas previamente registrados são aceitos."""
    ferramentas: dict[str, Callable[..., dict[str, str]]] = {
        "buscar_ticket": buscar_ticket,
    }

    ferramenta = ferramentas.get(tool_call.name)
    if ferramenta is None:
        raise ValueError(f"Ferramenta não permitida: {tool_call.name}")

    return ToolResult(name=tool_call.name, output=ferramenta(**tool_call.arguments))


def gerar_resposta_final_demo(mensagem_usuario: str, contexto: list[dict[str, Any]]) -> str:
    """Gera uma resposta didática usando apenas resultados presentes no contexto."""
    resultados = [item["output"] for item in contexto if item.get("type") == "tool_result"]

    if not resultados:
        return "Não encontrei um ticket no contexto. Informe um identificador como INC-2026-001."

    ticket = resultados[-1]
    if ticket.get("error") == "ticket_not_found":
        return f"O ticket {ticket['ticket_id']} não foi encontrado."

    prioridade = ticket["priority"]
    recomendacao = (
        "Escalar para a equipe responsável pelo serviço."
        if prioridade in {"high", "critical"} or ticket["service"] == "billing"
        else "Registrar o acompanhamento e investigar dentro da fila normal."
    )

    return (
        f"Ticket {ticket['ticket_id']}: {ticket['title']}\n"
        f"Serviço: {ticket['service']} | Prioridade atual: {prioridade}\n"
        f"Resumo: {ticket['description']}\n"
        f"Recomendação: {recomendacao}"
    )


def executar_agente_demo(mensagem_usuario: str) -> dict[str, Any]:
    """Executa o ciclo: decidir → chamar ferramenta → receber resultado → responder."""
    contexto: list[dict[str, Any]] = []
    rastreio: list[dict[str, Any]] = []

    proxima_acao = decidir_proxima_acao_demo(mensagem_usuario, contexto)

    while proxima_acao is not None:
        rastreio.append({"step": "tool_requested", **asdict(proxima_acao)})
        resultado = executar_tool_call(proxima_acao)
        contexto.append({"type": "tool_result", "name": resultado.name, "output": resultado.output})
        rastreio.append({"step": "tool_completed", "name": resultado.name, "output": resultado.output})
        proxima_acao = decidir_proxima_acao_demo(mensagem_usuario, contexto)

    resposta = gerar_resposta_final_demo(mensagem_usuario, contexto)
    rastreio.append({"step": "final_response"})

    return {
        "response": resposta,
        "trace": rastreio,
    }


resultado_demo = executar_agente_demo("Analise o ticket INC-2026-002. Ele precisa ser escalado?")
resultado_demo


{'response': 'Ticket INC-2026-002: Cobrança duplicada em pagamentos\nServiço: billing | Prioridade atual: medium\nResumo: Alguns clientes identificaram duas cobranças para a mesma transação.\nRecomendação: Escalar para a equipe responsável pelo serviço.',
 'trace': [{'step': 'tool_requested',
   'name': 'buscar_ticket',
   'arguments': {'ticket_id': 'INC-2026-002'}},
  {'step': 'tool_completed',
   'name': 'buscar_ticket',
   'output': {'ticket_id': 'INC-2026-002',
    'title': 'Cobrança duplicada em pagamentos',
    'service': 'billing',
    'description': 'Alguns clientes identificaram duas cobranças para a mesma transação.',
    'priority': 'medium',
    'status': 'open'}},
  {'step': 'final_response'}]}

### Leitura do rastreio

Observe o campo `trace` da célula anterior. Ele mostra algo que não existia no workflow linear:

- a primeira ação não é uma consulta feita automaticamente pelo código principal;
- primeiro há uma intenção de chamada: `tool_requested`;
- depois a aplicação executa apenas a ferramenta permitida: `tool_completed`;
- somente então vem a resposta final.

Mesmo neste modo simulado, fica visível a separação de responsabilidades:

| Responsabilidade | Quem executa? |
|---|---|
| Identificar que um ticket precisa ser consultado | política demonstrativa / modelo no modo real |
| Validar o nome da ferramenta | aplicação |
| Executar a consulta | aplicação |
| Devolver apenas os dados autorizados | ferramenta |
| Construir uma resposta para o usuário | política demonstrativa / modelo no modo real |


## 6. Agente real com LangChain e `@tool` (opcional)

Nesta seção, a função funcional `buscar_ticket` vira uma ferramenta reconhecida pelo framework. O decorator `@tool` não muda a regra de negócio: ele fornece metadados para que o modelo conheça o nome, a descrição e o schema de entrada.

A docstring e as anotações de tipo são parte da interface da ferramenta. Por isso, devem ser tratadas como contrato de produto, não como comentário decorativo.

> Esta célula só deve ser executada quando `OPENAI_API_KEY` e `OPENAI_MODEL` estiverem configurados e as dependências opcionais instaladas.


In [13]:
!pip install langchain


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

@tool
def buscar_ticket_tool(ticket_id: str) -> dict[str, str]:
    """Recupera um ticket de suporte pelo identificador.

    Use esta ferramenta antes de analisar ou recomendar ações sobre um ticket
    específico. Ela retorna título, serviço, descrição, prioridade e status.
    """
    return buscar_ticket(ticket_id)

instrucoes_do_agente = """
Você é um agente de triagem de incidentes.

Regras:
- Quando o usuário mencionar um identificador de ticket, use buscar_ticket_tool
    antes de responder sobre o conteúdo, risco ou encaminhamento desse ticket.
- Não invente dados que não estejam na mensagem do usuário ou no resultado da ferramenta.
- Se a ferramenta informar ticket_not_found, comunique isso claramente.
- Você pode recomendar, mas não pode atualizar, apagar ou escalar tickets diretamente.
- Ao final, apresente: resumo, prioridade observada e recomendação objetiva.
"""

modelo = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
agente = create_agent(
    model=modelo,
    tools=[buscar_ticket_tool],
    system_prompt=instrucoes_do_agente,
)

resultado_real = agente.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Analise o ticket INC-2026-002. Ele precisa ser escalado?",
        }
    ]
})

print(resultado_real["messages"][-1].content)


O ticket INC-2026-002 refere-se a um problema de cobrança duplicada em pagamentos, onde alguns clientes foram cobrados duas vezes pela mesma transação. O serviço afetado é o de billing, a prioridade está como média e o status é aberto.

Resumo: Cobrança duplicada em pagamentos, afetando clientes, prioridade média, status aberto.
Prioridade observada: Média.
Recomendação: Considerando que o problema impacta diretamente os clientes e envolve questões financeiras, é importante que o ticket seja tratado com atenção. Se a equipe atual não tiver capacidade ou especialização para resolver rapidamente, a escalada para um time especializado em billing ou financeiro pode ser recomendada para acelerar a resolução e evitar insatisfação dos clientes.


### Inspecionando as etapas do agente real

A chamada `.invoke(...)` devolve o estado final, incluindo as mensagens do usuário, do modelo e, quando houver, os resultados de ferramentas.

A próxima célula imprime uma versão resumida do histórico. Ela é útil para demonstrar que o agente pode ter produzido uma `tool_call` antes da resposta final.


In [16]:
if "resultado_real" not in globals():
    print("Execute a célula anterior após configurar as dependências e as variáveis de ambiente.")
else:
    for indice, mensagem in enumerate(resultado_real["messages"]):
        print(f"\n--- Mensagem {indice} | {type(mensagem).__name__} ---")

        if getattr(mensagem, "tool_calls", None):
            print("Tool calls:", mensagem.tool_calls)

        if getattr(mensagem, "content", None):
            print("Conteúdo:", mensagem.content)



--- Mensagem 0 | HumanMessage ---
Conteúdo: Analise o ticket INC-2026-002. Ele precisa ser escalado?

--- Mensagem 1 | AIMessage ---
Tool calls: [{'name': 'buscar_ticket_tool', 'args': {'ticket_id': 'INC-2026-002'}, 'id': 'call_KyBl0Y25BaU9jLcwxgd0ysGT', 'type': 'tool_call'}]

--- Mensagem 2 | ToolMessage ---
Conteúdo: {"ticket_id": "INC-2026-002", "title": "Cobrança duplicada em pagamentos", "service": "billing", "description": "Alguns clientes identificaram duas cobranças para a mesma transação.", "priority": "medium", "status": "open"}

--- Mensagem 3 | AIMessage ---
Conteúdo: O ticket INC-2026-002 refere-se a um problema de cobrança duplicada em pagamentos, onde alguns clientes foram cobrados duas vezes pela mesma transação. O serviço afetado é o de billing, a prioridade está como média e o status é aberto.

Resumo: Cobrança duplicada em pagamentos, afetando clientes, prioridade média, status aberto.
Prioridade observada: Média.
Recomendação: Considerando que o problema impacta di

## 7. Representação equivalente no Langflow

No Langflow, um agente recebe ferramentas pela entrada **Tools**. A ferramenta não precisa enviar dados automaticamente: ela fica disponível para ser chamada pelo agente conforme a necessidade da conversa.

A arquitetura visual desejada é:

```text
Chat Input ───────→ Agent ───────→ Chat Output
                       ↑
                       │ Tools
                       │
            Buscar Ticket / MCP Tools
```

Há duas formas didáticas de conectar essa capacidade ao canvas:

### Opção A — componente customizado

É a forma mais direta para uma demonstração local. O Langflow exige uma classe como adaptador para criar um componente customizado e habilitar **Tool Mode**. A lógica de negócio, porém, pode continuar fora da classe, em funções como `normalizar_ticket_id` e `buscar_ticket`.

A classe não deve conter a regra de consulta; ela apenas recebe a entrada do canvas e chama a função funcional.

### Opção B — MCP Tools

A função `buscar_ticket` pode ser exposta por um servidor MCP. No canvas, o componente `MCP Tools` conecta esse servidor ao `Agent`. Essa opção preserva melhor a separação entre a aplicação que disponibiliza a ferramenta e a aplicação Langflow que a consome.

Para a Aula 1, a Opção A é mais simples de demonstrar visualmente. Para uma aplicação maior, MCP é uma ponte útil para discutir interoperabilidade e governança de ferramentas.


### Passo a passo no canvas do Langflow

1. Adicione `Chat Input`, `Agent` e `Chat Output`.
2. Configure um modelo no componente `Agent`.
3. Adicione a fonte da ferramenta:
   - um componente customizado de busca de ticket; **ou**
   - um `MCP Tools` conectado a um servidor que exponha `buscar_ticket`.
4. Habilite **Tool Mode** na fonte da ferramenta.
5. Conecte a saída **Toolset/Tool** da fonte à entrada **Tools** do `Agent`.
6. Edite o nome e a descrição da ação para algo inequívoco, como:

```text
Nome: buscar_ticket
Descrição: Recupera título, serviço, descrição, prioridade e status de um ticket pelo ID.
Use antes de analisar um ticket identificado como INC-AAAA-NNN.
```

7. Nas instruções do agente, imponha a regra de consulta antes da análise.
8. Teste no Playground com: `Analise o ticket INC-2026-001.`

O elemento que faz uma ferramenta ser útil ao agente não é apenas a conexão visual: é a qualidade do seu contrato de entrada, da descrição e das permissões que ela efetivamente concede.


## 8. Segurança e confiabilidade: o que continua fora do alcance do agente

Adicionar uma ferramenta não justifica delegar decisões críticas ao modelo. O padrão seguro é:

```text
modelo sugere → código valida → sistema executa (ou pede aprovação humana)
```

Para este caso de triagem, o agente pode consultar e recomendar. Ele **não** deve poder:

- alterar prioridade ou status do ticket;
- acionar produção, reiniciar serviços ou executar comandos de infraestrutura;
- consultar tickets fora do escopo de autorização do usuário;
- recuperar campos confidenciais por padrão;
- definir sozinho que uma ação irreversível deve ocorrer.

Uma evolução segura seria separar ferramentas de leitura e escrita:

```text
buscar_ticket(ticket_id)                 # leitura: potencialmente automática
sugerir_escalonamento(ticket_id, motivo) # sugestão: ainda sem efeito externo
escalar_ticket(ticket_id, motivo)        # escrita: exige validação e aprovação humana
```

A pergunta de arquitetura deixa de ser “o agente consegue fazer?” e passa a ser “qual capacidade ele deve receber, em quais condições e com qual trilha de auditoria?”.


In [17]:
def validar_pedido_de_escalonamento(ticket_id: str, motivo: str) -> dict[str, str]:
    """Exemplo de guarda determinística antes de qualquer ação externa."""
    ticket = buscar_ticket(ticket_id)

    if ticket.get("error") == "ticket_not_found":
        return {"approved": "false", "reason": "ticket_not_found"}

    if len(motivo.strip()) < 20:
        return {
            "approved": "false",
            "reason": "justification_too_short",
        }

    return {
        "approved": "true",
        "reason": "eligible_for_human_review",
    }


validar_pedido_de_escalonamento(
    "INC-2026-002",
    "Há suspeita de impacto financeiro por cobrança duplicada.",
)


{'approved': 'true', 'reason': 'eligible_for_human_review'}

## 9. Comparação final: workflow determinístico versus agente com ferramenta

| Aspecto | Workflow linear | Agente com tool calling |
|---|---|---|
| Quem escolhe buscar o ticket? | código da aplicação | modelo, dentro de capacidades autorizadas |
| Em que momento ocorre a busca? | antes da chamada ao LLM | durante o ciclo de execução |
| Ordem das etapas | fixa | parcialmente dinâmica |
| Previsibilidade | maior | menor |
| Flexibilidade para perguntas variadas | menor | maior |
| Custo e latência | normalmente menores | podem aumentar a cada chamada adicional |
| Melhor cenário de uso | dados necessários são conhecidos | a necessidade de consulta depende da pergunta |

A principal conclusão não é que agentes substituem workflows lineares. É que cada arquitetura resolve um tipo de incerteza diferente.

- Use o workflow linear quando o caminho é conhecido.
- Use um agente quando a escolha entre fontes ou ações autorizadas depende do contexto.
- Em ambos os casos, mantenha contratos, validações, limites de permissão e rastreabilidade.


## 10. Exercícios

1. Adicione uma segunda ferramenta funcional, `buscar_status_servico(service)`, e atualize as instruções do agente para consultá-la apenas quando o ticket indicar erro de login ou indisponibilidade.
2. Faça `buscar_ticket` devolver apenas campos mínimos e discuta o que muda na capacidade de análise do agente.
3. Crie um teste que garanta que `buscar_ticket("inc-2026-001")` e `buscar_ticket("INC-2026-001")` devolvem o mesmo resultado.
4. Modifique o modo demonstração para chamar duas ferramentas em sequência.
5. Discuta quais critérios determinísticos deveriam bloquear um pedido de escalonamento antes de qualquer integração de escrita.
6. No Langflow, altere a descrição da ação para ser vaga e observe como isso pode piorar a seleção da ferramenta pelo agente.

### Fechamento

Este notebook adiciona um primeiro ciclo agentic ao material da Aula 1, mas não introduz multiagentes. A próxima etapa natural é discutir orquestração: várias responsabilidades, ferramentas diferentes, estado compartilhado e critérios claros para decidir quando essa complexidade é realmente necessária.
